# Graphene Defect Detector — end-to-end demo

Walks through the full pipeline on the small set of samples committed under `data/`:

1. Render `.xyz` structures to bond-map PNGs
2. Detect defects with YOLOv8 and crop them
3. Segment each crop into a defect mask
4. Extract geometric shape features
5. Aggregate per sample and merge the `total_energy` label
6. Train and evaluate the regressor

See `configs/pipeline.yaml` for the directory layout this notebook uses, and
`docs/known_quirks.md` for two intentionally-preserved quirks in the feature math.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
logging.basicConfig(level=logging.INFO)

from pathlib import Path
import pandas as pd

from graphene_defect_detector.config import PipelineConfig
from graphene_defect_detector import rendering, detection, segmentation, dataset, modeling, plots

config = PipelineConfig.from_yaml("../configs/pipeline.yaml")
config.ensure_output_dirs()
config

## 1. Render a few `.xyz` structures

In [ ]:
rendered = rendering.render_dataset(config.xyz_dir, config.renders_dir, config.render_resolution, limit=20)
rendered[:5]

## 2. Detect and crop defects with YOLOv8

Uses the pretrained weights in `models/defect_detector_yolov8.pt`. Requires `ultralytics` + `torch`.

In [ ]:
model = detection.load_model(config.weights_path)
crops = detection.crop_detections(config.renders_dir, config.crops_dir, model)
len(crops)

## 3. Segment each crop into a defect mask

In [ ]:
masks = segmentation.crops_to_masks(config.crops_dir, config.masks_dir)
len(masks)

## 4-5. Extract shape features, aggregate per sample, merge `total_energy`

This step runs over every mask currently in `data/masks/` (the full committed set, not just
the 20 samples rendered above) so the regressor below trains on the same ~2000-sample
dataset as the original thesis.

In [ ]:
defect_table, sample_table, energy_table = dataset.build_modeling_dataset(
    config.masks_dir, config.xyz_dir, config.energy_csv, contours_dir=config.contours_dir,
)
print(defect_table.shape, sample_table.shape, energy_table.shape)
energy_table.head()

In [ ]:
plots.plot_correlation_heatmap(energy_table)

In [ ]:
plots.plot_correlogram(energy_table)

## 6. Train and evaluate the regressor

In [ ]:
trained = modeling.train_regressor(energy_table, test_size=0.1, random_state=13)
metrics = modeling.evaluate(trained)
metrics

In [ ]:
plots.plot_training_deviance(trained)

In [ ]:
plots.plot_feature_importance(trained.model, trained.feature_columns)

In [ ]:
plots.plot_predictions_vs_actual(trained)

Saving the trained model for reuse (e.g. from the `gdd train` CLI):

In [ ]:
trained.save("../models/regressor.joblib")